# Final Notebook - Customer Segmentation Clustering

Notebook này chỉ dùng **hình ảnh, bảng tóm tắt và giải thích**, không chứa code cell. Mục tiêu là giúp kiểm soát toàn bộ chương trình và model một cách trực quan như một bản báo cáo Lab 3 hoàn chỉnh.

Dataset chính để chấm bài là **Mall Customers** vì có đúng các feature trong đề: `Age`, `Gender`, `Annual Income (k$)`, `Spending Score (1-100)`. Dataset **UCI Online Retail** là phần mở rộng raw transaction data để tạo RFM segmentation, không merge trực tiếp với Mall Customers vì không cùng customer ID/schema.


## 1. Define Problem

### Problem Statement

Ta có dữ liệu khách hàng của một cửa hàng bán lẻ gồm tuổi, giới tính, thu nhập hằng năm và spending score. Nhiệm vụ là **phân cụm khách hàng theo mức độ tương đồng** để tìm ra các customer segments khác biệt.

### Machine Learning Type

- Đây là bài toán **Unsupervised Learning**.
- Không có target label.
- Không dùng classification hoặc regression.
- Model không dự đoán nhãn có sẵn; model tự tìm cấu trúc cụm dựa trên feature similarity.

### Expected Output

- Số cụm hợp lý.
- Bảng profile từng cụm.
- Tên segment dễ hiểu.
- Hình scatter, dendrogram, heatmap để kiểm soát trực quan.
- Metric đánh giá clustering: silhouette, Davies-Bouldin, Calinski-Harabasz.


## 2. Data Sources

Chương trình sử dụng hai nguồn raw data:

| source_name | url | access_method | download_date | local_file |
| --- | --- | --- | --- | --- |
| kaggle_primary | https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python | kagglehub.dataset_download | 2026-07-01 | C:/Users/User/OneDrive - ut.edu.vn/Documents/Machine Learning - Trường/customer_segmentation_clustering/data/raw/Mall_Customers.csv |
| kaggle_primary | https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python | kagglehub.dataset_download | 2026-07-01 | C:/Users/User/OneDrive - ut.edu.vn/Documents/Machine Learning - Trường/customer_segmentation_clustering/data/raw/mall_customers/Mall_Customers.csv |
| uci_online_retail | https://archive.ics.uci.edu/dataset/352/online%2Bretail | direct_zip_download | 2026-07-01 | C:/Users/User/OneDrive - ut.edu.vn/Documents/Machine Learning - Trường/customer_segmentation_clustering/data/raw/online_retail/Online Retail.xlsx |

**Quy tắc quan trọng:** Mall Customers là dataset chính đúng đề bài. Online Retail chỉ là track mở rộng RFM, không merge theo dòng với Mall Customers.


## 3. Data Cleaning Summary

### Mall Customers

- Raw shape: `[200, 5]`
- Clean shape: `[200, 5]`
- Missing values raw: `{'CustomerID': 0, 'Gender': 0, 'Age': 0, 'Annual Income (k$)': 0, 'Spending Score (1-100)': 0}`
- IQR outlier counts: `{'age': 0, 'annual_income_k': 2, 'spending_score': 0}`

### Online Retail

- Raw shape: `[541909, 8]`
- Clean shape: `[397884, 9]`
- Removed rows: `144025`
- Issue counts: `{'cancellation': 9288, 'missing_customer_id': 135080, 'non_positive_quantity': 10624, 'non_positive_unit_price': 2517, 'malformed_invoice_date': 0}`
- Customer count after cleaning: `4338`


## 4. Mall Customers EDA

Các hình dưới đây giải thích dữ liệu chính trước khi train model. Đây là phần cần kiểm soát để biết dữ liệu có missing/outlier/phân bố bất thường hay không.


### Age Distribution

![Age Distribution](../reports/figures/age_histogram.png)

Histogram này cho biết phân bố độ tuổi của khách hàng trong Mall Customers. Đây là biến nhân khẩu học chính để xem nhóm khách hàng trẻ, trung niên hoặc lớn tuổi có hành vi chi tiêu khác nhau hay không.


### Annual Income Distribution

![Annual Income Distribution](../reports/figures/annual_income_k_histogram.png)

Biểu đồ thể hiện phân bố thu nhập hằng năm theo đơn vị nghìn USD. Thu nhập là một feature quan trọng khi phân cụm vì nó thường liên quan đến khả năng chi tiêu.


### Spending Score Distribution

![Spending Score Distribution](../reports/figures/spending_score_histogram.png)

Spending score thể hiện mức độ chi tiêu hoặc mức độ tương tác mua sắm. Phân bố biến này giúp nhận diện nhóm chi tiêu thấp, trung bình và cao.


### Numerical Feature Boxplots

![Numerical Feature Boxplots](../reports/figures/numeric_boxplots.png)

Boxplot dùng để kiểm tra outlier của age, annual income và spending score. Theo rule của chương trình, outlier được báo cáo nhưng không xóa tự động.


### Gender Distribution

![Gender Distribution](../reports/figures/gender_distribution.png)

Biểu đồ đếm số lượng khách hàng theo gender. Gender không phải feature bắt buộc trong mô hình cuối, nhưng được thử trong feature set numeric-plus-gender để so sánh.


### Correlation Heatmap

![Correlation Heatmap](../reports/figures/correlation_heatmap.png)

Heatmap correlation giúp kiểm tra quan hệ tuyến tính giữa các biến numeric. Vì clustering dựa trên khoảng cách, việc hiểu quan hệ giữa các feature giúp diễn giải cụm tốt hơn.


### Income vs Spending Scatter

![Income vs Spending Scatter](../reports/figures/income_vs_spending_scatter.png)

Scatter plot giữa annual income và spending score là hình quan trọng nhất cho bài toán customer segmentation, vì nó thường thể hiện rõ các nhóm như high income-high spending hoặc high income-low spending.


### Age vs Spending Scatter

![Age vs Spending Scatter](../reports/figures/age_vs_spending_scatter.png)

Biểu đồ age vs spending score giúp xem nhóm tuổi nào có xu hướng chi tiêu cao hoặc thấp.


### Age vs Income Scatter

![Age vs Income Scatter](../reports/figures/age_vs_income_scatter.png)

Biểu đồ age vs annual income hỗ trợ kiểm tra xem thu nhập có thay đổi theo độ tuổi trong dataset hay không.


## 5. Feature Scaling and Model Training

Clustering dựa trên khoảng cách, vì vậy các feature numeric được chuẩn hóa bằng `StandardScaler`.

Feature sets được so sánh:

- `numeric_only`: `age`, `annual_income_k`, `spending_score`.
- `numeric_plus_gender`: numeric features + one-hot encoded gender.

Algorithms được train:

- K-Means.
- Agglomerative/Hierarchical Clustering.
- DBSCAN.

Model cuối không được chọn chỉ theo metric. Chương trình ưu tiên model có metric tốt nhưng vẫn dễ giải thích và không bỏ quá nhiều khách hàng vào noise.


## 6. Best Mall Customers Model

| Metric | Value |
| --- | --- |
| Algorithm | kmeans |
| Feature set | numeric_only |
| Parameters | {"n_clusters": 6, "random_state": 42} |
| Number of clusters | 6 |
| Silhouette score | 0.4311 |
| Davies-Bouldin score | 0.835 |
| Calinski-Harabasz score | 134.4752 |

### Mall Customers Cluster Profile

| cluster_id | size | percentage | mean_age | median_age | mean_annual_income_k | median_annual_income_k | mean_spending_score | median_spending_score | top_gender | segment_name | business_interpretation |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 45 | 22.5 | 56.3333 | 54 | 54.2667 | 54 | 49.0667 | 49 | Female | Average Income - Average Spending | Cluster 0 contains 22.5% of customers. The segment has an average age of 56.3, average annual income of 54.3k USD, and average spending score of 49.1. The dominant gender category is Female. |
| 1 | 39 | 19.5 | 32.6923 | 32 | 86.5385 | 79 | 82.1282 | 83 | Female | High Income - High Spending | Cluster 1 contains 19.5% of customers. The segment has an average age of 32.7, average annual income of 86.5k USD, and average spending score of 82.1. The dominant gender category is Female. |
| 2 | 25 | 12.5 | 25.56 | 24 | 26.48 | 25 | 76.24 | 76 | Female | Young High Spenders | Cluster 2 contains 12.5% of customers. The segment has an average age of 25.6, average annual income of 26.5k USD, and average spending score of 76.2. The dominant gender category is Female. |
| 3 | 40 | 20 | 26.125 | 25 | 59.425 | 60 | 44.45 | 48 | Female | Average Income - Average Spending | Cluster 3 contains 20.0% of customers. The segment has an average age of 26.1, average annual income of 59.4k USD, and average spending score of 44.5. The dominant gender category is Female. |
| 4 | 30 | 15 | 44 | 43.5 | 90.1333 | 87 | 17.9333 | 16.5 | Male | High Income - Low Spending | Cluster 4 contains 15.0% of customers. The segment has an average age of 44.0, average annual income of 90.1k USD, and average spending score of 17.9. The dominant gender category is Male. |
| 5 | 21 | 10.5 | 45.5238 | 46 | 26.2857 | 25 | 19.381 | 15 | Female | Low Income - Low Spending | Cluster 5 contains 10.5% of customers. The segment has an average age of 45.5, average annual income of 26.3k USD, and average spending score of 19.4. The dominant gender category is Female. |


## 7. Mall Customers Model Evaluation Figures

Các hình dưới đây dùng để kiểm soát quá trình chọn số cụm, so sánh model và giải thích kết quả phân cụm cuối.


### K-Means Elbow Plot

![K-Means Elbow Plot](../reports/figures/kmeans_elbow_plot.png)

Elbow plot theo dõi inertia khi thay đổi số cụm k. Inertia giảm khi k tăng, nhưng điểm gấp khúc giúp chọn số cụm hợp lý hơn thay vì chọn quá nhiều cụm.


### Silhouette Score Comparison

![Silhouette Score Comparison](../reports/figures/silhouette_score_comparison.png)

Biểu đồ so sánh silhouette score giữa các ứng viên model. Silhouette càng cao thì cụm càng tách biệt và gọn hơn, nhưng vẫn cần cân bằng với khả năng giải thích.


### Hierarchical Dendrogram

![Hierarchical Dendrogram](../reports/figures/hierarchical_dendrogram.png)

Dendrogram minh họa quá trình gộp cụm của hierarchical clustering. Hình này giúp kiểm soát trực quan số cụm và khoảng cách giữa các nhóm.


### Best Clusters: Income vs Spending

![Best Clusters: Income vs Spending](../reports/figures/best_clusters_income_vs_spending.png)

Đây là hình diễn giải chính của best model trên Mall Customers. Màu sắc biểu thị cluster đã gán cho từng khách hàng.


### Best Clusters: Age vs Spending

![Best Clusters: Age vs Spending](../reports/figures/best_clusters_age_vs_spending.png)

Hình này giúp giải thích cluster theo độ tuổi và spending score, bổ sung góc nhìn nhân khẩu học cho biểu đồ income-spending.


### Cluster Size Distribution

![Cluster Size Distribution](../reports/figures/cluster_size_distribution.png)

Biểu đồ cho biết kích thước từng cluster. Nếu một cụm quá nhỏ hoặc quá lớn, cần kiểm tra lại tính ổn định và ý nghĩa thực tế.


### Cluster Profile Heatmap

![Cluster Profile Heatmap](../reports/figures/cluster_profile_heatmap.png)

Heatmap profile tóm tắt giá trị trung bình của các feature theo từng cluster, giúp đặt tên và diễn giải customer segment.


## 8. Mall Segment Interpretation

- `Average Income - Average Spending`: nhóm khách hàng trung bình, phù hợp cho chiến lược duy trì.
- `High Income - High Spending`: nhóm giá trị cao, có thể ưu tiên chăm sóc hoặc loyalty program.
- `Young High Spenders`: nhóm trẻ có spending score cao, phù hợp chiến dịch sản phẩm mới hoặc promotion.
- `High Income - Low Spending`: nhóm có tiềm năng nhưng chưa chi tiêu nhiều, cần phân tích động lực mua hàng.
- `Low Income - Low Spending`: nhóm chi tiêu thấp, phù hợp ưu đãi nhỏ hoặc sản phẩm phổ thông.


## 9. Online Retail RFM Extension

Track Online Retail dùng raw transaction data để tạo RFM features:

- `recency_days`: số ngày từ lần mua gần nhất.
- `frequency`: số invoice duy nhất.
- `monetary_value`: tổng giá trị mua hàng.
- `average_order_value`: giá trị trung bình mỗi invoice.

Track này chứng minh chương trình có thể mở rộng sang dữ liệu giao dịch thật, nhưng kết quả chính của đề vẫn là Mall Customers.


## 10. Best Online Retail RFM Model

| Metric | Value |
| --- | --- |
| Algorithm | agglomerative |
| Feature set | rfm |
| Parameters | {"linkage": "single", "n_clusters": 3} |
| Number of clusters | 3 |
| Silhouette score | 0.9473 |
| Davies-Bouldin score | 0.1334 |
| Calinski-Harabasz score | 943.7181 |

### Online Retail RFM Cluster Profile

| cluster_id | size | percentage | mean_recency_days | mean_frequency | mean_monetary_value | mean_average_order_value | segment_name | business_interpretation |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 2 | 0.0461 | 163.5 | 1.5 | 122828.05 | 80709.925 | At-Risk Low-Frequency Customers | Cluster 0 contains 0.0% of customers. Mean recency is 163.5 days, mean frequency is 1.5 invoices, and mean monetary value is 122828.1. |
| 1 | 1 | 0.0231 | 1 | 201 | 143825.06 | 715.5476 | Recent High-Value Customers | Cluster 1 contains 0.0% of customers. Mean recency is 1.0 days, mean frequency is 201.0 invoices, and mean monetary value is 143825.1. |
| 2 | 4335 | 99.9308 | 92.5248 | 4.2279 | 1965.8424 | 382.0549 | Frequent High-Value Customers | Cluster 2 contains 99.9% of customers. Mean recency is 92.5 days, mean frequency is 4.2 invoices, and mean monetary value is 1965.8. |


### RFM Recency Distribution

![RFM Recency Distribution](../reports/online_retail/figures/recency_days_histogram.png)

Recency cho biết số ngày từ lần mua gần nhất đến snapshot date. Recency thấp thường biểu thị khách hàng còn hoạt động gần đây.


### RFM Frequency Distribution

![RFM Frequency Distribution](../reports/online_retail/figures/frequency_histogram.png)

Frequency là số hóa đơn duy nhất của mỗi khách hàng. Đây là chỉ báo hành vi mua lặp lại.


### RFM Monetary Distribution

![RFM Monetary Distribution](../reports/online_retail/figures/monetary_value_histogram.png)

Monetary value là tổng giá trị mua hàng của khách hàng sau khi làm sạch giao dịch.


### Average Order Value Distribution

![Average Order Value Distribution](../reports/online_retail/figures/average_order_value_histogram.png)

Average order value bổ sung góc nhìn về giá trị trung bình mỗi đơn hàng, tránh chỉ nhìn tổng monetary.


### RFM Correlation Heatmap

![RFM Correlation Heatmap](../reports/online_retail/figures/rfm_correlation_heatmap.png)

Heatmap kiểm tra quan hệ giữa recency, frequency, monetary value và average order value.


### Recency vs Monetary Scatter

![Recency vs Monetary Scatter](../reports/online_retail/figures/recency_vs_monetary_scatter.png)

Scatter plot này giúp nhận diện nhóm khách hàng mua gần đây và có giá trị cao.


### RFM K-Means Elbow Plot

![RFM K-Means Elbow Plot](../reports/online_retail/figures/rfm_kmeans_elbow_plot.png)

Elbow plot cho RFM track giúp đánh giá số cụm hợp lý khi dùng K-Means trên dữ liệu giao dịch.


### RFM Silhouette Score Comparison

![RFM Silhouette Score Comparison](../reports/online_retail/figures/rfm_silhouette_score_comparison.png)

Biểu đồ so sánh silhouette cho các ứng viên RFM clustering.


### RFM Hierarchical Dendrogram

![RFM Hierarchical Dendrogram](../reports/online_retail/figures/rfm_hierarchical_dendrogram.png)

Dendrogram cho RFM track, dùng để kiểm tra cấu trúc phân cấp của khách hàng giao dịch.


### RFM Clusters: Recency vs Monetary

![RFM Clusters: Recency vs Monetary](../reports/online_retail/figures/rfm_clusters_recency_vs_monetary.png)

Hình phân cụm chính cho RFM track, biểu diễn cluster theo recency và monetary value.


### RFM Cluster Size Distribution

![RFM Cluster Size Distribution](../reports/online_retail/figures/rfm_cluster_size_distribution.png)

Biểu đồ kích thước cluster trên Online Retail RFM.


### RFM Cluster Profile Heatmap

![RFM Cluster Profile Heatmap](../reports/online_retail/figures/cluster_profile_heatmap.png)

Heatmap profile của RFM clusters, dùng để diễn giải nhóm khách hàng theo recency, frequency và monetary.


## 11. Program and Model Summary

### Source Structure

- `src/data_acquisition.py`: tải raw data Mall Customers và UCI Online Retail.
- `src/data_quality.py`: làm sạch và validate Mall Customers.
- `src/online_retail.py`: làm sạch transaction và tạo RFM features.
- `src/preprocessing.py`: scale feature sets.
- `src/clustering.py`: train K-Means, Agglomerative, DBSCAN.
- `src/evaluation.py`: tính metric và chọn model cuối.
- `src/model_train.py`: lưu model artifact `.joblib`.
- `app.py`: Streamlit UI chỉ đọc artifact, không retrain.

### Model Artifacts

- `models/customer_segmentation_pipeline.joblib`
- `models/online_retail_segmentation_pipeline.joblib`

### Validation Commands

- `python -m compileall -q config.py main.py app.py src tests`
- `python -m pytest -q`
- `python -m json.tool notebooks/99_customer_segmentation_workflow.ipynb`


## 12. Final Conclusion

Chương trình đã đáp ứng các yêu cầu:

- Tìm kiếm thêm raw data.
- Lọc sạch data.
- Train clustering models.
- Viết tests cho data/model/UI/notebook.
- Có giao diện Python bằng Streamlit.
- Có notebook cuối chỉ chứa hình ảnh và giải thích để kiểm soát chương trình/model.

Kết quả chính để trình bày là **Mall Customers segmentation**. Online Retail RFM là phần mở rộng để tăng độ đầy đủ về raw data và workflow thực tế.
